# EDA — Supplier & Supply-Chain Analytics

Notebook này dùng để **trình bày kết quả**, không chứa logic ETL.
Toàn bộ logic nằm trong `src/` và `sql/`. Chạy `python run_all.py` trước.

> ⚠️ Dữ liệu supplier / product / shipment là **SYNTHETIC**.
> Và dataset gốc cũng là dữ liệu tổng hợp. Mọi kết quả dưới đây minh hoạ
> **phương pháp**, không phải insight thị trường thật.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

DB = Path("..") / "data" / "processed" / "supply_chain.db"
FIGURES = Path("..") / "reports" / "figures"
conn = sqlite3.connect(DB)

tables = pd.read_sql_query(
    "SELECT name, type FROM sqlite_master WHERE type IN ('table','view') "
    "AND name NOT LIKE 'sqlite_%' ORDER BY name",
    conn,
)
tables["rows"] = [
    conn.execute(f"SELECT COUNT(*) FROM {n}").fetchone()[0] for n in tables["name"]
]
print(f"Database: {DB.resolve()}")
tables

## Mảng 1 — Hiệu suất nhà cung cấp

35 supplier trong danh bạ, 33 đang hoạt động. Tier dự đoán được chất lượng.

In [ ]:
cols = [
    "supplier_name", "tier", "country", "category_focus", "is_active",
    "order_count", "total_spend", "abc_class", "on_time_pct", "defect_pct",
    "avg_lead_days", "xyz_class",
]
pd.read_sql_query(
    f"SELECT {', '.join(cols)} FROM mart_supplier_scorecard "
    "WHERE is_active = 1 ORDER BY total_spend DESC LIMIT 10",
    conn,
)

In [ ]:
display(Image(filename=str(FIGURES / "02_supplier_quality_map.png")))

## Mảng 2 — Rủi ro phụ thuộc nhà cung cấp

Category nào có 1 supplier chiếm >50% chi tiêu?

In [ ]:
pd.read_sql_query(
    "SELECT category, supplier_name, tier, category_share_pct, "
    "suppliers_in_category, products_without_backup "
    "FROM mart_supplier_concentration WHERE is_single_source_risk = 1 "
    "ORDER BY category_share_pct DESC",
    conn,
)

## Mảng 6–8 — Kiểm chứng dữ liệu gốc là nhiễu

Ba mảng dùng dữ liệu gốc đều cho kết quả **phẳng**. Đây là kết luận trung thực.

In [ ]:
print("--- Payment (tỷ trọng đơn gần như bằng nhau) ---")
display(pd.read_sql_query(
    "SELECT cc_provider, order_count, order_share_pct, avg_order_value "
    "FROM mart_payment_profile ORDER BY order_share_pct DESC", conn))

print("\n--- Buyer persona (đơn TB chỉ lệch nhẹ) ---")
display(pd.read_sql_query(
    "SELECT job_family, order_count, total_spend, avg_order_value "
    "FROM mart_buyer_persona ORDER BY total_spend DESC", conn))

## Chất lượng dữ liệu

20/20 quy tắc ĐẠT — bao gồm kiểm tra `purchase_price` không bị sửa.

In [ ]:
dq = pd.read_sql_query(
    "SELECT severity, COUNT(*) AS rules, SUM(failed_rows) AS total_failed "
    "FROM mart_data_quality GROUP BY severity",
    conn,
)
display(dq)
display(pd.read_sql_query(
    "SELECT rule_name, failed_rows, pass_rate, note FROM mart_data_quality "
    "WHERE severity = 'high' ORDER BY rule_name", conn))

In [ ]:
conn.close()